# GINN Loss Function Low-Frequency Well QC

Use the low-frequency `log_ai_anchor_time.npz` bundle to test how candidate regularization terms respond to controlled log-AI perturbations. This notebook is diagnostic only: it does not modify `src/` and does not rerun GINN training.

Output directory: `scripts/output/ginn_loss_function_lowfreq_well_qc_<timestamp>/`


In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

# Global plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

REPO = Path(r"C:\Users\WangQinZhuo\Program\xihu_workflow_standardlize")
TRAIN_YAML = REPO / "experiments/ginn/results/2026060901/train.yaml"
ANCHOR_NPZ = REPO / "scripts/output/well_constraints_20260608_184057/log_ai_anchor_time.npz"
WAVELET_CSV = REPO / "scripts/output/wavelet_generation_20260608_183012/selected_wavelet.csv"

print(f"REPO: {REPO}")
print(f"ANCHOR_NPZ exists: {ANCHOR_NPZ.exists()}")
print(f"WAVELET_CSV exists: {WAVELET_CSV.exists()}")

In [ ]:
# =======================================================================
# Data loading
# =======================================================================

def load_anchor_bundle(path: Path) -> dict:
    with np.load(path, allow_pickle=False) as z:
        out = {key: z[key].copy() for key in z.files}
    for key in ("schema_version", "sample_domain", "sample_unit", "summary_json", "metadata_json"):
        if key in out and np.asarray(out[key]).shape == ():
            value = np.asarray(out[key]).item()
            if isinstance(value, bytes):
                value = value.decode("utf-8")
            out[key] = value
    for key in ("summary_json", "metadata_json"):
        if key in out and isinstance(out[key], str):
            out[key] = json.loads(out[key])
    return out


bundle = load_anchor_bundle(ANCHOR_NPZ)
samples = bundle["samples"].astype(float)
dt_s = float(np.median(np.diff(samples)))
target_log_ai = bundle["target_log_ai"].astype(float)
target_ai = bundle["target_ai"].astype(float)
mask = bundle["anchor_mask"].astype(bool)
weight = bundle["anchor_weight"].astype(float)
well_names = bundle["anchor_names"].astype(str)
anchor_types = bundle["anchor_types"].astype(str)

assert bundle["sample_domain"] == "time", bundle["sample_domain"]
assert bundle["sample_unit"] in ("s", "sec", "second", "seconds"), bundle["sample_unit"]
assert target_log_ai.shape == mask.shape == weight.shape

n_anchors = mask.shape[0]
n_sample = mask.shape[1]

print(f"Loaded {n_anchors} anchors x {n_sample} samples")
print(f"dt_s = {dt_s:.6f} s,  duration = {samples[-1] - samples[0]:.3f} s")
print(f"Wells: {well_names.tolist()}")
print(f"Anchor types: {anchor_types.tolist()}")
for i, name in enumerate(well_names):
    n_valid = int(np.count_nonzero(mask[i]))
    print(f"  {name}: {n_valid} valid samples")

In [ ]:
# =======================================================================
# Load wavelet
# =======================================================================

wavelet_df = pd.read_csv(WAVELET_CSV)
wavelet_time = wavelet_df["time_s"].to_numpy(float)
wavelet = wavelet_df["amplitude"].to_numpy(float)
wavelet_dt = float(np.median(np.diff(wavelet_time)))
assert abs(wavelet_dt - dt_s) < 1e-6, f"dt mismatch: wavelet={wavelet_dt}, samples={dt_s}"

print(f"Wavelet: {len(wavelet)} samples, dt={wavelet_dt:.4f}s, "
      f"t=[{wavelet_time[0]:.3f}, {wavelet_time[-1]:.3f}]s")

# Quick wavelet plot
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(wavelet_time, wavelet, lw=0.8)
ax.axhline(0, color="gray", lw=0.5)
ax.set(xlabel="Time (s)", ylabel="Amplitude", title="Selected Wavelet")
plt.tight_layout()
plt.show()

In [ ]:
# Forward model (NumPy, same logic as src/ginn/physics.py)

def reflectivity(ai: np.ndarray) -> np.ndarray:
    """AI -> reflectivity, shape (T,) -> (T,), last sample padded with 0."""
    upper = ai[:-1]
    lower = ai[1:]
    r = (lower - upper) / (lower + upper + 1e-10)
    return np.r_[r, 0.0]


def convolve_reflectivity(r: np.ndarray, wavelet: np.ndarray) -> np.ndarray:
    """Convolve reflectivity with wavelet, output same length as r.

    Mirrors `ForwardModel.convolve()` in src/ginn/physics.py: asymmetric
    padding based on wavelet argmax, then convolution. `np.convolve` with the
    original wavelet is equivalent to PyTorch conv1d with the flipped kernel.
    """
    i_max = int(np.argmax(np.abs(wavelet)))
    pad_left = i_max
    pad_right = len(wavelet) - 1 - i_max
    padded = np.pad(r, (pad_left, pad_right), mode="constant", constant_values=0.0)
    return np.convolve(padded, wavelet, mode="valid")


def synthetic_from_log_ai(log_ai: np.ndarray, wavelet: np.ndarray) -> np.ndarray:
    """log(AI) -> AI -> reflectivity -> synthetic seismic."""
    ai = np.exp(log_ai)
    r = reflectivity(ai)
    return convolve_reflectivity(r, wavelet)


# Valid segment extraction

def valid_segments(valid: np.ndarray, min_len: int = 32) -> list[slice]:
    """Return contiguous valid segments with at least `min_len` samples."""
    idx = np.flatnonzero(valid)
    if idx.size == 0:
        return []
    breaks = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends = np.r_[idx[breaks], idx[-1]]
    return [slice(int(s), int(e) + 1) for s, e in zip(starts, ends) if e - s + 1 >= min_len]


def longest_valid_segment(valid: np.ndarray, min_len: int = 32) -> slice | None:
    """Return the longest contiguous valid segment, or None if no segment qualifies."""
    segs = valid_segments(valid, min_len=min_len)
    if not segs:
        return None
    return max(segs, key=lambda s: s.stop - s.start)


def center_valid(x: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Remove the valid-window mean while leaving invalid samples unchanged."""
    y = x.copy().astype(float)
    if np.any(valid):
        y[valid] -= float(np.mean(y[valid]))
    return y


def weighted_mean(x: np.ndarray, w: np.ndarray) -> float:
    """Weighted mean, NaN-safe."""
    xx = np.asarray(x, dtype=float)
    ww = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    finite = np.isfinite(xx)
    ww = np.where(finite, ww, 0.0)
    denom = float(np.sum(ww))
    if denom <= 0:
        return float("nan")
    return float(np.sum(np.where(finite, xx, 0.0) * ww) / denom)


In [ ]:
# Loss functions (all operate on residual = candidate - base, inside valid mask)

def l2_residual(residual, valid, w):
    """Per-sample L2 (mean squared residual)."""
    return weighted_mean(residual[valid] ** 2, w[valid])


def tv1(residual, valid, w):
    """First-order Total Variation: mean(|r[t+1] - r[t]|)."""
    d = residual[1:] - residual[:-1]
    pair_valid = valid[1:] & valid[:-1]
    pair_w = np.minimum(w[1:], w[:-1])
    return weighted_mean(np.abs(d[pair_valid]), pair_w[pair_valid])


def tv2(residual, valid, w):
    """Second-order TV: mean(|r[t+2] - 2*r[t+1] + r[t]|)."""
    d2 = residual[2:] - 2 * residual[1:-1] + residual[:-2]
    tri_valid = valid[2:] & valid[1:-1] & valid[:-2]
    tri_w = np.minimum(np.minimum(w[2:], w[1:-1]), w[:-2])
    return weighted_mean(np.abs(d2[tri_valid]), tri_w[tri_valid])


def huber_abs(x, delta):
    """Huber loss on absolute values: quadratic below delta, linear above."""
    ax = np.abs(x)
    return np.where(ax <= delta, 0.5 * ax * ax / delta, ax - 0.5 * delta)


def tv2_huber(residual, valid, w, delta=0.005):
    """Huber-smoothed second-order TV."""
    d2 = residual[2:] - 2 * residual[1:-1] + residual[:-2]
    tri_valid = valid[2:] & valid[1:-1] & valid[:-2]
    tri_w = np.minimum(np.minimum(w[2:], w[1:-1]), w[:-2])
    return weighted_mean(huber_abs(d2[tri_valid], delta), tri_w[tri_valid])


def spectral_high_energy(residual, valid, dt_s, cutoff_hz=70.0):
    """Fraction of tapered valid-segment spectral energy above cutoff_hz."""
    seg = longest_valid_segment(valid, min_len=8)
    if seg is None:
        return float("nan")
    x = np.asarray(residual[seg], dtype=float)
    if x.size < 8 or not np.any(np.isfinite(x)):
        return float("nan")
    x = np.where(np.isfinite(x), x, 0.0)
    x = x - float(np.mean(x))
    if x.size >= 16:
        x = x * np.hanning(x.size)
    spec = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(x.size, d=dt_s)
    power = np.abs(spec) ** 2
    total = float(np.sum(power))
    if total <= 0:
        return 0.0
    return float(np.sum(power[freqs >= cutoff_hz]) / total)


def alternating_score(residual, valid, w=None):
    """Fraction of sign flips in adjacent first differences.

    Not a formal loss; it is only an indicator. Sawtooth should be near 1.0,
    while smooth perturbations should be near 0.0.
    """
    d = residual[1:] - residual[:-1]
    pair_valid = valid[1:] & valid[:-1]
    s = np.sign(d[pair_valid])
    s = s[s != 0]
    if s.size < 3:
        return float("nan")
    flips = s[1:] * s[:-1] < 0
    return float(np.mean(flips))


# Registry for uniform iteration. `alternating_score` is logged as a diagnostic,
# but is not a proposed training loss.
LOSS_FUNCTIONS = {
    "l2_residual": l2_residual,
    "tv1": tv1,
    "tv2": tv2,
    "tv2_huber": tv2_huber,
    "spectral_high_energy_70hz": lambda r, v, w: spectral_high_energy(r, v, dt_s, cutoff_hz=70.0),
    "alternating_score": alternating_score,
}


In [ ]:
# Perturbation candidates

RESIDUAL_RMS_LEVELS = [0.005, 0.01, 0.02, 0.04]
CANDIDATE_NAMES = [
    "zero",
    "smooth_sine_low",
    "single_step",
    "blocky_layers",
    "alternating_saw",
    "high_sine_near_nyquist",
    "random_highpass_noise",
]


def scale_to_rms(x: np.ndarray, target_rms: float, valid: np.ndarray) -> np.ndarray:
    """Scale array so that RMS inside valid mask equals target_rms."""
    y = center_valid(x, valid)
    values = y[valid]
    rms = float(np.sqrt(np.mean(values ** 2))) if values.size else 0.0
    if rms <= 0 or not np.isfinite(rms):
        return np.zeros_like(y)
    return y * (target_rms / rms)


def _put_segment(values: np.ndarray, seg: slice, n: int) -> np.ndarray:
    out = np.zeros(n, dtype=float)
    out[seg] = values
    return out


def make_residuals(n: int, dt_s: float, valid: np.ndarray, rms: float) -> dict[str, np.ndarray]:
    """Generate perturbation residuals at a target RMS level.

    Shapes are generated in local coordinates of the longest valid segment so
    step/block candidates always lie inside the evaluated well interval.
    """
    seg = longest_valid_segment(valid, min_len=32)
    if seg is None:
        return {name: np.zeros(n, dtype=float) for name in CANDIDATE_NAMES}

    local_n = seg.stop - seg.start
    local_t = np.arange(local_n) * dt_s
    out = {"zero": np.zeros(n, dtype=float)}

    out["smooth_sine_low"] = _put_segment(np.sin(2 * np.pi * 8.0 * local_t), seg, n)

    step_local = np.zeros(local_n, dtype=float)
    step_local[local_n // 2 :] = 1.0
    out["single_step"] = _put_segment(step_local, seg, n)

    blocks_local = np.zeros(local_n, dtype=float)
    block_width = max(4, local_n // 40)
    block_step = max(12, local_n // 8)
    for k in range(5, local_n, block_step):
        blocks_local[k : k + block_width] += -1.0 if (k // block_step) % 2 else 1.0
    out["blocky_layers"] = _put_segment(blocks_local, seg, n)

    alt_local = ((np.arange(local_n) % 2) * 2 - 1).astype(float)
    out["alternating_saw"] = _put_segment(alt_local, seg, n)

    high_freq_hz = 0.45 / dt_s
    out["high_sine_near_nyquist"] = _put_segment(np.sin(2 * np.pi * high_freq_hz * local_t), seg, n)

    rng = np.random.default_rng(20260609 + local_n)
    noise_local = rng.normal(size=local_n)
    try:
        sos = butter(4, 70.0, btype="highpass", fs=1.0 / dt_s, output="sos")
        noise_local = sosfiltfilt(sos, noise_local)
    except ValueError:
        noise_local = noise_local - np.mean(noise_local)
    out["random_highpass_noise"] = _put_segment(noise_local, seg, n)

    return {name: scale_to_rms(value, rms, valid) for name, value in out.items()}


def make_mixed_waveform_null(residuals: dict[str, np.ndarray], base_log_ai, valid, wavelet) -> np.ndarray | None:
    """Select the high-frequency perturbation with the smallest waveform MAE."""
    base_syn = synthetic_from_log_ai(base_log_ai, wavelet)
    candidates = []
    for name in ["alternating_saw", "high_sine_near_nyquist", "random_highpass_noise"]:
        r = residuals.get(name)
        if r is None:
            continue
        cand_log_ai = base_log_ai + r
        cand_syn = synthetic_from_log_ai(cand_log_ai, wavelet)
        mae = float(np.mean(np.abs(cand_syn[valid] - base_syn[valid])))
        tv2_val = tv2(r, valid, np.ones_like(r))
        spec_val = spectral_high_energy(r, valid, dt_s, cutoff_hz=70.0)
        candidates.append((name, r, mae, tv2_val, spec_val))

    if not candidates:
        return None
    candidates.sort(key=lambda x: x[2])
    return candidates[0][1].copy()


print("Perturbation functions defined.")
print(f"RMS levels: {RESIDUAL_RMS_LEVELS}")
print(f"Base candidates: {CANDIDATE_NAMES}")


In [ ]:
# =======================================================================
# Main computation: iterate over wells, perturbations, RMS levels
# =======================================================================

ALL_CANDIDATES = list(CANDIDATE_NAMES) + ["mixed_waveform_null"]
rows = []
by_well_rows = []
skipped_wells = []

for i_well in range(n_anchors):
    well_name = str(well_names[i_well])
    well_mask = mask[i_well].astype(bool)
    well_w = weight[i_well].astype(float)
    base = target_log_ai[i_well].astype(float)

    # Find longest valid segment
    segs = valid_segments(well_mask, min_len=32)
    if not segs:
        skipped_wells.append((well_name, "no_valid_segment"))
        print(f"SKIP {well_name}: no valid segment with >=32 samples")
        continue

    seg = longest_valid_segment(well_mask, min_len=32)
    if seg is None:
        continue
    seg_mask = np.zeros(n_sample, dtype=bool)
    seg_mask[seg] = True
    combined_valid = well_mask & seg_mask
    n_valid = int(np.count_nonzero(combined_valid))
    if n_valid < 32:
        skipped_wells.append((well_name, f"segment_too_short({n_valid})"))
        print(f"SKIP {well_name}: longest segment only {n_valid} samples")
        continue

    base_syn = synthetic_from_log_ai(base, wavelet)
    base_seg_rms = float(np.sqrt(np.mean(base[combined_valid] ** 2)))
    print(f"\n{'='*60}")
    print(f"Well: {well_name}  |  valid samples: {n_valid}  |  base RMS: {base_seg_rms:.4f}")

    for rms_level in RESIDUAL_RMS_LEVELS:
        residuals = make_residuals(n_sample, dt_s, combined_valid, rms_level)
        mixed = make_mixed_waveform_null(residuals, base, combined_valid, wavelet)
        if mixed is not None:
            residuals["mixed_waveform_null"] = mixed
        else:
            # Fallback: copy alternating_saw
            residuals["mixed_waveform_null"] = residuals["alternating_saw"].copy()

        for cand_name in ALL_CANDIDATES:
            r = residuals.get(cand_name)
            if r is None:
                continue
            cand_log_ai = base + r
            cand_ai = np.exp(cand_log_ai)

            # Waveform misfit
            cand_syn = synthetic_from_log_ai(cand_log_ai, wavelet)
            waveform_mae = float(np.mean(np.abs(cand_syn[combined_valid] - base_syn[combined_valid])))

            # Residual statistics inside valid segment
            r_valid = r[combined_valid]
            residual_rms = float(np.sqrt(np.mean(r_valid ** 2)))
            residual_mae = float(np.mean(np.abs(r_valid)))

            # All loss candidates
            scores = {}
            for loss_name, loss_fn in LOSS_FUNCTIONS.items():
                val = loss_fn(r, combined_valid, well_w)
                scores[loss_name] = float(val) if not np.isnan(val) else float("nan")

            rows.append({
                "well_name": well_name,
                "n_valid": n_valid,
                "dt_s": dt_s,
                "candidate": cand_name,
                "residual_rms": residual_rms,
                "residual_mae": residual_mae,
                "waveform_mae_to_base": waveform_mae,
                **scores,
            })

    print(f"  rows accumulated so far: {len(rows)}")

# =======================================================================
# Build summary DataFrame
# =======================================================================
df_scores = pd.DataFrame(rows)
print(f"\nTotal rows: {len(df_scores)}")
print(f"Wells: {df_scores['well_name'].nunique()}")
print(f"Candidates: {df_scores['candidate'].unique().tolist()}")
print(f"RMS levels: {sorted(df_scores['residual_rms'].round(4).unique())}")
if skipped_wells:
    print(f"Skipped wells: {skipped_wells}")
df_scores.head(10)

In [ ]:
# =======================================================================
# By-well aggregation: ranking ratios
# =======================================================================

LOSS_NAMES_FOR_RANK = ["tv1", "tv2", "tv2_huber", "spectral_high_energy_70hz"]

by_well = []
for well in df_scores["well_name"].unique():
    for rms_level in RESIDUAL_RMS_LEVELS:
        sub = df_scores[(df_scores["well_name"] == well) &
                        (df_scores["residual_rms"].round(4) == round(rms_level, 4))]
        if sub.empty:
            continue

        def _get_loss(cand_name, loss_name):
            row = sub[sub["candidate"] == cand_name]
            if row.empty:
                return float("nan")
            return float(row[loss_name].iloc[0])

        for loss_name in LOSS_NAMES_FOR_RANK:
            alt = _get_loss("alternating_saw", loss_name)
            step = _get_loss("single_step", loss_name)
            blk = _get_loss("blocky_layers", loss_name)
            hs = _get_loss("high_sine_near_nyquist", loss_name)
            ss = _get_loss("smooth_sine_low", loss_name)

            by_well.append({
                "well_name": well,
                "loss_name": loss_name,
                "rms_level": round(rms_level, 4),
                "alternating_over_single_step": alt / step if (step and step > 0) else float("nan"),
                "alternating_over_blocky_layers": alt / blk if (blk and blk > 0) else float("nan"),
                "high_sine_over_smooth_sine": hs / ss if (ss and ss > 0) else float("nan"),
                "alternating_raw": alt,
                "single_step_raw": step,
            })

df_by_well = pd.DataFrame(by_well)
print(f"By-well rows: {len(df_by_well)}")
print(f"\nPer-loss summary of alternating_saw / single_step ratio (higher = better discrimination):")
summary = df_by_well.groupby("loss_name")["alternating_over_single_step"].agg(["mean", "median", "std", "count"])
summary = summary.round(3)
print(summary.to_string())

print(f"\nPer-loss summary of high_sine / smooth_sine ratio:")
summary2 = df_by_well.groupby("loss_name")["high_sine_over_smooth_sine"].agg(["mean", "median", "std", "count"])
print(summary2.round(3).to_string())
df_by_well.head(10)

In [ ]:
# =======================================================================
# Figure 1: Well curve perturbation examples
# =======================================================================

# Select 1-2 wells with the most valid samples for visualisation
well_sample_counts = df_scores.groupby("well_name")["n_valid"].max().sort_values(ascending=False)
plot_wells = well_sample_counts.index[:2].tolist()
PLOT_RMS = 0.02  # Fixed RMS level for visualisation
PLOT_CANDIDATES = [
    "single_step",
    "blocky_layers",
    "alternating_saw",
    "high_sine_near_nyquist",
]

n_plot_wells = len(plot_wells)
fig, axes = plt.subplots(n_plot_wells, len(PLOT_CANDIDATES) + 1,
                         figsize=(4 * (len(PLOT_CANDIDATES) + 1), 3 * n_plot_wells),
                         squeeze=False)

for row_idx, well_name in enumerate(plot_wells):
    i_well = list(well_names).index(well_name)
    well_mask = mask[i_well].astype(bool)
    base = target_log_ai[i_well].astype(float)

    segs = valid_segments(well_mask, min_len=32)
    seg = longest_valid_segment(well_mask, min_len=32)
    if seg is None:
        continue
    seg_mask = np.zeros(n_sample, dtype=bool)
    seg_mask[seg] = True
    valid = well_mask & seg_mask

    residuals = make_residuals(n_sample, dt_s, valid, PLOT_RMS)
    t_seg = samples[seg]
    base_seg = np.exp(base[seg])

    # Base curve
    ax = axes[row_idx, 0]
    ax.plot(t_seg, base_seg, color="gray", lw=1.0, label="base LFM AI")
    ax.set(xlabel="TWT (s)", ylabel="AI", title=f"{well_name}\nBase (LFM)")
    ax.legend(fontsize=8)

    for col_idx, cand_name in enumerate(PLOT_CANDIDATES):
        ax = axes[row_idx, col_idx + 1]
        r = residuals[cand_name]
        cand_log_ai = np.clip(base + r, -10, 20)
        cand_ai = np.exp(cand_log_ai)
        ax.plot(t_seg, base_seg, color="gray", lw=0.8, alpha=0.5, label="base")
        ax.plot(t_seg, cand_ai[seg], color="C1", lw=0.8, label="perturbed")
        ax.set(xlabel="TWT (s)", ylabel="AI",
               title=f"{well_name}: {cand_name}\n(rms={PLOT_RMS})")
        ax.legend(fontsize=8)

fig.suptitle("Well Curve Perturbation Examples", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =======================================================================
# Figure 2: Loss rank heatmap
# =======================================================================

# Compute z-score per loss function across all candidate x well x RMS combinations
# to make different loss scales comparable
LOSS_METRICS = ["tv1", "tv2", "tv2_huber", "spectral_high_energy_70hz", "l2_residual"]
DISPLAY_CANDS = ["single_step", "blocky_layers", "alternating_saw",
                 "high_sine_near_nyquist", "random_highpass_noise",
                 "smooth_sine_low", "mixed_waveform_null"]

# Build z-score matrix
heatmap_data = {}
for loss_name in LOSS_METRICS:
    vals = df_scores[loss_name].values
    finite = np.isfinite(vals)
    if not np.any(finite):
        heatmap_data[loss_name] = {c: float("nan") for c in DISPLAY_CANDS}
        continue
    mean_val = np.mean(vals[finite])
    std_val = np.std(vals[finite])
    if std_val <= 0:
        std_val = 1.0
    for cand in DISPLAY_CANDS:
        sub = df_scores[df_scores["candidate"] == cand][loss_name]
        sub_finite = sub[np.isfinite(sub)]
        if sub_finite.empty:
            heatmap_data.setdefault(loss_name, {})[cand] = float("nan")
        else:
            heatmap_data.setdefault(loss_name, {})[cand] = float(np.mean(sub_finite) - mean_val) / std_val

heatmap_matrix = pd.DataFrame(heatmap_data).T[DISPLAY_CANDS]

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(heatmap_matrix.values, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)

ax.set_xticks(range(len(DISPLAY_CANDS)))
ax.set_xticklabels(DISPLAY_CANDS, rotation=30, ha="right")
ax.set_yticks(range(len(LOSS_METRICS)))
ax.set_yticklabels(LOSS_METRICS)
ax.set_title("Loss Function Sensitivity Heatmap (z-score)")
ax.set_xlabel("Perturbation Candidate")
ax.set_ylabel("Loss Function")

# Annotate cells
for i in range(len(LOSS_METRICS)):
    for j in range(len(DISPLAY_CANDS)):
        val = heatmap_matrix.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color="black" if abs(val) < 1.0 else "white")

plt.colorbar(im, ax=ax, label="z-score", shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3: Waveform null-space scatter

SCATTER_CANDS = [
    "single_step",
    "blocky_layers",
    "alternating_saw",
    "high_sine_near_nyquist",
    "random_highpass_noise",
    "smooth_sine_low",
    "mixed_waveform_null",
]
PLOT_EPS = 1e-12

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_loss in zip(axes, ["tv2", "spectral_high_energy_70hz"]):
    for cand in SCATTER_CANDS:
        sub = df_scores[df_scores["candidate"] == cand]
        if sub.empty:
            continue
        x = sub["waveform_mae_to_base"].to_numpy(float)
        y = sub[y_loss].to_numpy(float)
        finite = np.isfinite(x) & np.isfinite(y)
        x_plot = np.maximum(x[finite], PLOT_EPS)
        y_plot = np.maximum(y[finite], PLOT_EPS) if y_loss == "spectral_high_energy_70hz" else y[finite]
        ax.scatter(x_plot, y_plot, alpha=0.5, s=20, label=cand)

    ax.set_xlabel("Waveform MAE to Base")
    ax.set_ylabel(y_loss)
    ax.set_title(f"Nullspace: Waveform MAE vs {y_loss}")
    ax.legend(fontsize=7, loc="upper left")
    ax.set_xscale("log")
    if y_loss == "spectral_high_energy_70hz":
        ax.set_yscale("log")

fig.suptitle("Waveform Null-Space Diagnostics", fontsize=13)
plt.tight_layout()
plt.show()


## Conclusions & Recommendations

*Answer the five questions from the guide.*

> **Note:** The conclusions below are based on the output of the notebook above.  
> Re-run the notebook after any code or data changes to refresh these numbers.

### 1. Does first-order TV explain the "can't pull back sawtooth" problem?

*(Fill after reviewing the heatmap and ratio tables.)*

- Look at `alternating_score` for `alternating_saw`: should be near 1.0.
- Compare `tv1` on `alternating_saw` vs `single_step` - the ratio tells you whether TV1 discriminates well.
- If `alternating_over_single_step` for `tv1` is near 1.0, then TV1 **cannot** distinguish sawtooth from a real interface.

### 2. Is second-order TV better at separating sawtooth from real interfaces?

- Compare `tv2` ratios vs `tv1` ratios in `df_by_well`.
- Expected: `tv2.alternating_over_single_step` >> 1.0, significantly higher than `tv1`'s ratio.

### 3. Is Huber TV2 more robust than pure L1 TV2?

- Compare `tv2_huber` vs `tv2` ratios.
- Huber should reduce outlier sensitivity while maintaining discrimination.

### 4. Should spectral high-energy loss be added to training?

- Check the `waveform_nullspace_scatter.png`: low waveform MAE + high spectral energy = null-space ringing.
- Decision: if `spectral_high_energy_70hz` clearly separates null-space candidates, it may be worth adding.  
  If not, TV2 may be sufficient.

### 5. Recommended implementation

```text
Recommendation:
- Add configurable residual second-order TV as an additional term (tv2),
  not as a replacement for first-order TV (keep tv1 as an option).
- Suggested config fields: lambda_tv2 (float, default 0.0),
  tv2_huber_delta (float | null, default null).
- Suggested logged fields: residual_tv2, tv2_term,
  residual_tv2_huber (if huber enabled).
- Keep spectral loss as a follow-up experiment unless the notebook
  shows TV2 cannot separate null-space ringing.
```

In [ ]:
# =======================================================================
# Export results
# =======================================================================

timestamp = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = REPO / f"scripts/output/ginn_loss_function_lowfreq_well_qc_{timestamp}"
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# --- CSV exports ---
df_scores.to_csv(OUTPUT_DIR / "loss_candidate_scores.csv", index=False)
df_by_well.to_csv(OUTPUT_DIR / "loss_candidate_scores_by_well.csv", index=False)
print("  [ok] loss_candidate_scores.csv")
print("  [ok] loss_candidate_scores_by_well.csv")

# --- Figures ---
# Re-plot Figure 1 to file
n_plot_wells = len(plot_wells)
fig1, axes1 = plt.subplots(n_plot_wells, len(PLOT_CANDIDATES) + 1,
                           figsize=(4 * (len(PLOT_CANDIDATES) + 1), 3 * n_plot_wells),
                           squeeze=False)

for row_idx, well_name in enumerate(plot_wells):
    i_well = list(well_names).index(well_name)
    well_mask = mask[i_well].astype(bool)
    base = target_log_ai[i_well].astype(float)

    segs = valid_segments(well_mask, min_len=32)
    seg = longest_valid_segment(well_mask, min_len=32)
    if seg is None:
        continue
    seg_mask = np.zeros(n_sample, dtype=bool)
    seg_mask[seg] = True
    valid = well_mask & seg_mask

    residuals = make_residuals(n_sample, dt_s, valid, PLOT_RMS)
    t_seg = samples[seg]
    base_seg = np.exp(base[seg])

    ax = axes1[row_idx, 0]
    ax.plot(t_seg, base_seg, color="gray", lw=1.0, label="base LFM AI")
    ax.set(xlabel="TWT (s)", ylabel="AI", title=f"{well_name}\nBase (LFM)")
    ax.legend(fontsize=8)

    for col_idx, cand_name in enumerate(PLOT_CANDIDATES):
        ax = axes1[row_idx, col_idx + 1]
        r = residuals[cand_name]
        cand_log_ai = np.clip(base + r, -10, 20)
        cand_ai = np.exp(cand_log_ai)
        ax.plot(t_seg, base_seg, color="gray", lw=0.8, alpha=0.5, label="base")
        ax.plot(t_seg, cand_ai[seg], color="C1", lw=0.8, label="perturbed")
        ax.set(xlabel="TWT (s)", ylabel="AI",
               title=f"{well_name}: {cand_name}\n(rms={PLOT_RMS})")
        ax.legend(fontsize=8)

fig1.suptitle("Well Curve Perturbation Examples", fontsize=13, y=1.01)
fig1.tight_layout()
fig1.savefig(OUTPUT_DIR / "figures/well_curve_perturbation_examples.png",
             dpi=150, bbox_inches="tight")
plt.close(fig1)
print("  [ok] well_curve_perturbation_examples.png")

# Figure 2: heatmap
fig2, ax2 = plt.subplots(figsize=(10, 4.5))
im2 = ax2.imshow(heatmap_matrix.values, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
ax2.set_xticks(range(len(DISPLAY_CANDS)))
ax2.set_xticklabels(DISPLAY_CANDS, rotation=30, ha="right")
ax2.set_yticks(range(len(LOSS_METRICS)))
ax2.set_yticklabels(LOSS_METRICS)
ax2.set_title("Loss Function Sensitivity Heatmap (z-score)")
for i in range(len(LOSS_METRICS)):
    for j in range(len(DISPLAY_CANDS)):
        val = heatmap_matrix.iloc[i, j]
        if np.isfinite(val):
            ax2.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=8, color="black" if abs(val) < 1.0 else "white")
plt.colorbar(im2, ax=ax2, label="z-score", shrink=0.8)
fig2.tight_layout()
fig2.savefig(OUTPUT_DIR / "figures/loss_rank_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig2)
print("  [ok] loss_rank_heatmap.png")

# Figure 3: scatter
fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_loss in zip(axes3, ["tv2", "spectral_high_energy_70hz"]):
    for cand in SCATTER_CANDS:
        sub = df_scores[df_scores["candidate"] == cand]
        if sub.empty:
            continue
        x = sub["waveform_mae_to_base"].to_numpy(float)
        y = sub[y_loss].to_numpy(float)
        finite = np.isfinite(x) & np.isfinite(y)
        x_plot = np.maximum(x[finite], PLOT_EPS)
        y_plot = np.maximum(y[finite], PLOT_EPS) if y_loss == "spectral_high_energy_70hz" else y[finite]
        ax.scatter(x_plot, y_plot, alpha=0.5, s=20, label=cand)
    ax.set_xlabel("Waveform MAE to Base")
    ax.set_ylabel(y_loss)
    ax.set_title(f"Nullspace: Waveform MAE vs {y_loss}")
    ax.legend(fontsize=7, loc="upper left")
    ax.set_xscale("log")
    if y_loss == "spectral_high_energy_70hz":
        ax.set_yscale("log")
fig3.suptitle("Waveform Null-Space Diagnostics", fontsize=13)
fig3.tight_layout()
fig3.savefig(OUTPUT_DIR / "figures/waveform_nullspace_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig3)
print("  [ok] waveform_nullspace_scatter.png")

# --- Metadata ---
metadata = {
    "created_at_unix": time.time(),
    "repo": str(REPO),
    "train_yaml": str(TRAIN_YAML),
    "anchor_npz": str(ANCHOR_NPZ),
    "wavelet_csv": str(WAVELET_CSV),
    "n_wells": int(df_scores["well_name"].nunique()),
    "n_rows": len(df_scores),
    "rms_levels": RESIDUAL_RMS_LEVELS,
    "candidates_evaluated": sorted(df_scores["candidate"].unique().tolist()),
    "loss_functions_evaluated": sorted(LOSS_FUNCTIONS.keys()),
    "skipped_wells": skipped_wells,
    "dt_s": dt_s,
}
with (OUTPUT_DIR / "metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
print("  [ok] metadata.json")

print(f"\n[ok] All outputs written to: {OUTPUT_DIR}")